In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/content/huggingface_security_top500_scan_v2.csv')

In [3]:
STRONG_KEYWORDS = {
    "security", "vulnerability", "vulnerabilities",
    "adversarial", "attack", "attacks",
    "poison", "poisoning", "safety"
}

WEAK_KEYWORDS = {
    "eval", "evaluation", "limitations",
    "robust", "risk", "risks", "mitigation"
}

In [4]:
def classify_match(keywords_found):
    """Return 'strong' if at least one genuinely security-relevant
    keyword is present, otherwise 'weak'."""
    kws = set(eval(keywords_found)) if isinstance(keywords_found, str) else set(keywords_found)
    if kws & STRONG_KEYWORDS:
        return "strong"
    return "weak"

In [5]:
df["match_strength"] = df["keywords_found"].apply(classify_match)

In [6]:
strong = df[df["match_strength"] == "strong"]
weak   = df[df["match_strength"] == "weak"]

In [7]:
print(f"Total models from initial scan:       {len(df)}")
print(f"Strong (security-relevant) matches:   {len(strong)}")
print(f"Weak   (generic ML vocabulary) matches: {len(weak)}")
print()
print("Strong matches:")
for _, r in strong.iterrows():
    print(f"  {r['modelId']:55s}  keywords: {r['keywords_found']}")

Total models from initial scan:       218
Strong (security-relevant) matches:   15
Weak   (generic ML vocabulary) matches: 203

Strong matches:
  deepset/roberta-base-squad2                              keywords: ['adversarial']
  deepset/tinyroberta-squad2                               keywords: ['adversarial']
  PocketDoc/Dans-PersonalityEngine-V1.3.0-24b              keywords: ['safety']
  deepset/roberta-base-squad2-distilled                    keywords: ['adversarial']
  EleutherAI/deep-ignorance-pretraining-stage-unfiltered   keywords: ['safety']
  protectai/deberta-v3-base-prompt-injection               keywords: ['security']
  TheBloke/Mixtral-8x7B-Instruct-v0.1-AWQ                  keywords: ['safety']
  jackaduma/SecBERT                                        keywords: ['security']
  vicgalle/Configurable-Hermes-2-Pro-Llama-3-8B            keywords: ['eval', 'safety']
  deepset/deberta-v3-base-squad2                           keywords: ['adversarial']
  eliasalbouzidi/distilb

In [8]:
# Save both for traceability
df.to_csv("huggingface_security_all_with_strength.csv", index=False)
strong.to_csv("huggingface_security_strong_only.csv", index=False)